# PACKET INPUT

In [25]:
import pandas as pd
import numpy as np
from datetime import datetime
import json
import re
from datetime import datetime, timedelta
import os
#import pyarrow as pa

pd.set_option('display.float_format', '{:.6f}'.format)
pd.set_option('display.max_rows', 300)


In [27]:
def packet_to_df(packet_list, packet_type,line_number):
    ## Make this info a function, feed into function to make dfs
    if packet_type == 'ppg':
        packet_df = pd.DataFrame(packet_list)
        packet_matrix = packet_df.to_numpy().T
    else:
        packet_matrix = np.array(packet_list).T

    if packet_type == 'imu':
        cols = ['ax', 'ay', 'az', 'qx', 'qy', 'qz', 'qw']
    elif packet_type == 'ecg':
        cols = ['ecg']
    elif packet_type == 'ppg':
        cols = ['b', 'r', 'ir', 'g']
    elif packet_type == 'eda':
        cols = ['eda']
    elif packet_type == 'temperature':
        cols = ['temperature']
    sensor_df = pd.DataFrame(packet_matrix, columns = cols)
    sensor_df['line_number'] = line_number
    return sensor_df




In [30]:
#file = r'C:\Users\a_hop\Downloads\BIOPOINT\blackshort.txt'
#file = r'C:\Users\a_hop\Downloads\BIOPOINT\blackshort2.txt'
#file = r'C:\Users\a_hop\Downloads\BIOPOINT\whiteshort.txt'
#file = r'C:\Users\a_hop\Downloads\BIOPOINT\greenshort2.txt'
#file = r'C:\Users\a_hop\OneDrive - McGill University\UNI 2025\BIOPOINT Data\gas challenge save files\sub-0001\sub-0001-new_ses-1_task-gas_biopoint.txt'
#file = r'C:\Users\a_hop\OneDrive - McGill University\UNI 2025\BIOPOINT Data\ppg test files\sub-50Hz_ses-2_task-overnight_biopoint.txt'
#file = r'C:\Users\a_hop\Downloads\BIOPOINT\Exp2_NoEMG_LeftHand.txt'
#file = r'C:\Users\a_hop\Downloads\BIOPOINT\Exp3_NoEMG_RightHand.txt'
#file = r'C:\Users\a_hop\Downloads\BIOPOINT\Exp4_NoEMG_LeftHand.txt'
file = r'C:\Users\a_hop\Documents\A1 Biopoint Watch Tests\sub-blacktest3-ses-1_task-gas_biopoint.txt'
#file = r'C:\Users\a_hop\Documents\A1 Biopoint Watch Tests\sub-green_ses-1_task-short_biopoint.txt'

output_types = ['ecg', 'ppg', 'imu', 'temp']
line_number = 1
experiment_id = 1
packet_types = []
start_date = 0
imu_dfs = []
ppg_dfs = []
ecg_dfs = []
eda_dfs = []
temp_dfs = []

with open(file) as f:
    for packet_line in f:
        #Create json
        packet_line_fixed = packet_line.replace("'",'"').replace("None", "null").replace("True", "true").replace("False", "false")
        packet = json.loads(packet_line_fixed)
        packet_types.append(packet['packet_type'])
        #Initiate vars
        packet_list = []

        if packet['packet_type'] == 'start_time':
            #start_date = datetime(int(packet["data"]["year"][0]), int(packet["data"]["month"][0]), int(packet["data"]["day"][0]), int(packet["data"]["hours"][0]), int(packet["data"]["minutes"][0]), int(packet["data"]["seconds"][0])) 
            start_date = datetime(int(packet["data"]["year"][0]), int(packet["data"]["month"][0]), int(packet["data"]["day"][0]), int(packet["data"]["hour"][0]), int(packet["data"]["minute"][0]), int(packet["data"]["second"][0])) 
            continue
        if packet["packet_type"] == 'imu' and 'imu' in output_types:
            packet_list.append(packet["data"]["ax"])
            packet_list.append(packet["data"]["ay"])
            packet_list.append(packet["data"]["az"])
            packet_list.append(packet["data"]["qx"])
            packet_list.append(packet["data"]["qy"])
            packet_list.append(packet["data"]["qz"])
            packet_list.append(packet["data"]["qw"])
            imu_df  = packet_to_df(packet_list, packet['packet_type'], line_number)
            imu_dfs.append(imu_df)

        elif packet["packet_type"] == 'temperature' and 'temp' in output_types:
            packet_list.append(packet["data"]["temperature"])
            temp_df = packet_to_df(packet_list, packet['packet_type'], line_number)
            temp_dfs.append(temp_df)
        elif packet["packet_type"] == 'ecg' and 'ecg' in output_types:
            packet_list.append(packet["data"]["ecg"])
            ecg_df = packet_to_df(packet_list, packet['packet_type'], line_number)
            ecg_dfs.append(ecg_df)
        elif packet["packet_type"] == 'eda' and 'eda' in output_types:
            packet_list.append(packet["data"]["eda"])
            eda_df = packet_to_df(packet_list, packet['packet_type'], line_number)
            eda_dfs.append(eda_df)
        elif packet["packet_type"] == 'ppg' and 'ppg' in output_types:
            packet_list.append(packet["data"]["b"])
            packet_list.append(packet["data"]["r"])
            packet_list.append(packet["data"]["ir"])
            packet_list.append(packet["data"]["g"])
            ppg_df = packet_to_df(packet_list, packet['packet_type'], line_number)
            ppg_dfs.append(ppg_df)
        else:
            continue

        last_line = packet
        line_number = line_number + 1


In [31]:
distinct_packet_types = list(set(packet_types))
print(distinct_packet_types)


['start_time', 'ecg', 'start_packet', 'temperature', 'ppg', 'status', 'imu', 'memory']


In [32]:
for i in range(len(output_types)):
    output_type = output_types[i]
    if output_type == 'ecg':
        ecg_all_df = pd.concat(ecg_dfs, ignore_index = True)
        time_interval = 0.002
        number_samples = len(ecg_all_df)
        times = [start_date+timedelta(seconds = i*time_interval) for i in range(number_samples)]
        ecg_all_df['datetime'] = times
        ecg_all_df = ecg_all_df.drop(['Unnamed: 0', 'line_number'], axis=1, errors='ignore')
    elif output_type == 'temp':
        temp_all_df = pd.concat(temp_dfs, ignore_index = True)
        time_interval = 1
        number_samples = len(temp_all_df)
        times = [start_date+timedelta(seconds = i*time_interval) for i in range(number_samples)]
        temp_all_df['datetime'] = times
        temp_all_df = temp_all_df.drop(['Unnamed: 0', 'line_number'], axis=1, errors='ignore')
    elif output_type == 'ppg':
        ppg_all_df = pd.concat(ppg_dfs, ignore_index = True)
        for col in ppg_all_df.columns:
            non_nan = ppg_all_df[col].dropna().reset_index(drop=True)
            ppg_all_df[col] = non_nan.reindex(range(len(ppg_all_df)))
        ppg_all_df = ppg_all_df.dropna(subset=['ir','b','r','g'], how='all')
        time_interval = 0.02
        number_samples = len(ppg_all_df)
        times = [start_date+timedelta(seconds = i*time_interval) for i in range(number_samples)]
        ppg_all_df['datetime'] = times
        ppg_all_df = ppg_all_df.drop(['Unnamed: 0', 'line_number'], axis=1, errors='ignore')
    elif output_type == 'eda':
        eda_all_df = pd.concat(eda_dfs, ignore_index = True)
        time_interval = 0.02
        number_samples = len(eda_all_df)
        times = [start_date+timedelta(seconds = i*time_interval) for i in range(number_samples)]
        eda_all_df['datetime'] = times
        eda_all_df = eda_all_df.drop(['Unnamed: 0', 'line_number'], axis=1, errors='ignore')
    elif output_type == 'imu':
        imu_all_df = pd.concat(imu_dfs, ignore_index = True)
        number_samples = len(imu_all_df)
        time_interval = 0.01
        times = [start_date+timedelta(seconds = i*time_interval) for i in range(number_samples)]
        imu_all_df['datetime'] = times
        imu_all_df = imu_all_df.drop(['Unnamed: 0', 'line_number'], axis=1, errors='ignore')



In [35]:
temp_all_df.tail(1)

,temperature,datetime
112,29.207470,2026-06-10 14:25:11


In [37]:
output_folder = r'C:\Users\a_hop\Documents\A1 Biopoint Watch Tests\omgtest'
for i in range(len(output_types)):
    if output_types[i] == 'ecg':
        ecg_all_df.to_parquet(os.path.join(output_folder,'ecg.parquet'))
    elif output_types[i] == 'temp':
        temp_all_df.to_parquet(os.path.join(output_folder,'temp.parquet'))
    elif output_types[i] == 'ppg':
        ppg_all_df.to_parquet(os.path.join(output_folder,'ppg.parquet'))
    elif output_types[i] == 'eda':
        eda_all_df.to_parquet(os.path.join(output_folder,'eda.parquet'))
    elif output_types[i] == 'imu':
        imu_all_df.to_parquet(os.path.join(output_folder,'imu.parquet'))


# CSV INPUT

## Higher level - many files

In [3]:
# Loop
start_file_number = 17
end_file_number = 17
root_directory = r'C:\Users\a_hop\Downloads\BIOPOINT'
for folder in os.listdir(root_directory):
    folder_path = os.path.join(root_directory, folder)
    match_exp_number = re.search(r'(?<=Exp)\d+', folder)
    if match_exp_number:
        exp_number = match_exp_number.group()
        if int(exp_number)>= start_file_number and int(exp_number)<=end_file_number:
            raw_files_path = os.path.join(folder_path, "Raw Files")
            print(folder)
            files = [f for f in os.listdir(raw_files_path) if not f.lower().endswith('.txt')]
            files.sort()  # sort alphabetically
            for i, f in enumerate(files):
                file_path = os.path.join(raw_files_path, f)
                file_path = os.path.abspath(file_path)  # makes it absolute
                match_date = re.search(r'\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2}', file_path)
                if match_date:
                    date_str = match_date.group()
                    start_time = datetime.strptime(date_str,"%Y-%m-%d-%H-%M-%S")
                    if i == 0:
                        true_date = start_time
                else:
                    raise ValueError("No valid date found in filename: {file_path}")
                    break
                output_type = f.rsplit("_", 1)[-1].replace('.csv','')
                print(output_type)
                if start_time == true_date and output_type != 'emg':
                    csv_df = pd.read_csv(file_path)
                    if output_type == 'ecg':
                        time_interval = 0.002
                    elif output_type == 'temperature':
                        time_interval = 2.4
                    elif output_type == 'imu':
                        time_interval = 0.01
                    else:
                        time_interval = 0.02
                    number_samples = len(csv_df)
                    csv_df['datetime'] = pd.date_range(start = start_time, periods = number_samples, freq = f"{time_interval}s")
                    csv_df = csv_df.drop(columns = ["time"])
                    output_file = os.path.join(folder_path, output_type + '_output.parquet')
                    csv_df.to_parquet(output_file, index = False)
                    #csv_df.to_csv(output_file, index = False)
                else:
                    continue
                


            


Exp17_LeftSideBreathing
ecg
eda
emg
imu
ppg
temperature
temperature


## Lower Level - Single Files

In [50]:

#csv_file = r'C:\Users\a_hop\Downloads\BIOPOINT\Exp11_LeftForearm_Palmar\Raw Files\default-1_2025-10-29-13-12-26_temperature.csv'
csv_file = r'C:\Users\a_hop\Downloads\BIOPOINT\Exp5_LeftHand_Dorsal\Raw Files\default-1_2025-10-08-13-29-50_temperature.csv'
#DD
#Find Start Date
match_date = re.search(r'\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2}', csv_file)
if match_date:
    date_str = match_date.group()
    start_time = datetime.strptime(date_str,"%Y-%m-%d-%H-%M-%S")
    print(date_str)
    #print(start_time)
else:
    raise ValueError("No valid date found in filename: {csv_file}")




2025-10-08-13-29-50


In [ ]:
#Use if CSV is only file
csv_df = pd.read_csv(csv_file)
output_type ='temperature'
#Calculate DF
if output_type == 'ecg':
    time_interval = 0.002
elif output_type == 'temperature':
    #time_interval = 2.178
    time_interval = 2.4
elif output_type == 'imu':
    time_interval = 0.01
else:
    time_interval = 0.02
    
number_samples = len(csv_df)
csv_df['datetime'] = pd.date_range(start = start_time, periods = number_samples, freq = f"{time_interval}s")
csv_df = csv_df.drop(columns = ["time"])
#csv_df.to_parquet(r'C:\Users\a_hop\Downloads\BIOPOINT\Exp7_LeftHand_Palmar\temperature_output.parquet', index = False)
#csv_df.to_csv(r'C:\Users\a_hop\Downloads\BIOPOINT\Exp5_NoEMG_LeftHand\temperature_output.csv', index=False)



In [52]:
# I am stupid
csv_df.tail(1)

,temperature,datetime
21952,26.700001,2025-10-09 02:46:41.456
